# Appendix 1: Adding systematic uncertainties

Johann Brehmer, Felix Kling, Irina Espejo, and Kyle Cranmer 2018-2019

In this tutorial we'll explain how to add systematic uncertainties to the MadMiner workflow. Note that the treatment of systematic uncertainties changed substantially with `MadMiner v0.6`, including changes to the MadMiner file specification. Please don't use files from older MadMiner versions with systematic uncertainties.

## Preparations

Before you execute this notebook, make sure you have running installations of MadGraph, Pythia, and Delphes.

In [1]:
import subprocess
subprocess.check_call(["pip", "install", "-e", "/home/shared/madminer"])


Obtaining file:///home/shared/madminer
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for madminer (pyproject.toml): started
  Building editable for madminer (pyproject.toml): finished with status 'done'
  Created wheel for madminer: filename=madminer-0.9.7.dev24-py3-none-any.whl size=5682 sha256=4bb0abd2efb2bf07dda50affc1368e7f6019fa03c9ff5573a3e30b2a96349f49
  Stored in directory: /tmp/pip-ephem-wheel-cache-4e4q0qdr/whe


[notice] A new release of pip is available: 24.2 -> 25.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


0

In [16]:
import madminer
print("Location:", madminer.__file__)


Location: /home/shared/madminer/madminer/__init__.py


In [2]:
import os
os.environ["LD_LIBRARY_PATH"] = (
    "/madminer/software/MG5_aMC_v2_9_16/HEPTools/lhapdf6_py3/lib:"
    + os.environ.get("LD_LIBRARY_PATH", "")
)


In [3]:
import os
import logging
import numpy as np

from madminer.core import MadMiner
from madminer.lhe import LHEReader
from madminer.sampling import SampleAugmenter
from madminer import sampling
from madminer.ml import ScoreEstimator
from madminer.fisherinformation import FisherInformation, profile_information, project_information
from madminer.plotting import plot_systematics, plot_fisher_information_contours_2d
from particle import Particle

/home/shared/madminer/madminer/fisherinformation/geometry.py:349: SyntaxWarning: "is not" with a literal. Did you mean "!="?
  if self.dimension is not 2:


Please enter here the environment variable pointing to your MG5 installation folder.

In [4]:
mg_dir = os.getenv("MG_FOLDER_PATH")

MadMiner uses the Python `logging` module to provide additional information and debugging output. You can choose how much of this output you want to see by switching the level in the following lines to `logging.DEBUG` or `logging.WARNING`.

In [5]:
# MadMiner output
logging.basicConfig(
    format="%(asctime)-5.5s %(name)-20.20s %(levelname)-7.7s %(message)s",
    datefmt="%H:%M",
    level=logging.DEBUG,
)

# Output of all other modules (e.g. matplotlib)
for key in logging.Logger.manager.loggerDict:
    if "madminer" not in key:
        logging.getLogger(key).setLevel(logging.WARNING)

## 1. Parameters and benchmarks

We'll just load the MadMiner setup from the first part of this tutorial:

In [6]:
miner = MadMiner()
miner.load("data/setup.h5")

15:14 madminer.utils.inter INFO    HDF5 file does not contain nuisance parameters information
15:14 madminer.utils.inter INFO    HDF5 file does not contain finite difference information
15:14 madminer.utils.inter INFO    HDF5 file does not contain observables information
15:14 madminer.utils.inter INFO    HDF5 file does not contain sample summary information
15:14 madminer.utils.inter INFO    HDF5 file does not contain sample information
15:14 madminer.utils.inter INFO    HDF5 file does not contain systematic information
15:14 madminer.core.madmin INFO    Found 2 parameters:
15:14 madminer.core.madmin INFO       CWL2 (LHA: dim6 2, Power: 2, Range: (-20.0, 20.0))
15:14 madminer.core.madmin INFO       CPWL2 (LHA: dim6 5, Power: 2, Range: (-20.0, 20.0))
15:14 madminer.core.madmin INFO    Found 6 benchmarks:
15:14 madminer.core.madmin INFO       sm: CWL2 = 0.00e+00, CPWL2 = 0.00e+00
15:14 madminer.core.madmin INFO       w: CWL2 = 15.20, CPWL2 = 0.10
15:14 madminer.core.madmin INFO       ne

## 2. Set up systematics, save settings

This is where things become interesting: We want to model systematic uncertainties. The main function is `add_systematics()`, the keyword `effect` determines how the effect of the nuisance parameters on the event weights is calculated. For `effect="norm"`, the nuisance parameter rescales thee overall cross section of one or multiple samples. For `effect="pdf"`, its effect is calculated with PDF variations. Finally, with `effect="scale"` scale variations are used.

Here we consider three nuisance parameters: one for the signal normalization, one for the background normalization, and one for scale uncertainties (which we here assume to be correlated between signal and background).

In [7]:
# miner.add_systematics(effect="norm", systematic_name="signal_norm", norm_variation=1.1)
# miner.add_systematics(effect="norm", systematic_name="bkg_norm", norm_variation=1.2)
miner.add_systematics(effect="scale", systematic_name="scales", scale="mu")

Again, we save our setup:

In [8]:
miner.save("data/setup_systematics.h5")

15:14 madminer.core.madmin INFO    Saving setup (including morphing) to data/setup_systematics.h5


## 3. Run MadGraph

In [9]:
import os
os.environ["LD_LIBRARY_PATH"] = (
    "/madminer/software/MG5_aMC_v2_9_16/HEPTools/lhapdf6_py3/lib:"
    + os.environ.get("LD_LIBRARY_PATH", "")
)


Now it's time to run MadGraph. MadMiner will instruct MadGraph to use its built-in `systematics` tool to calculate how the event weights change under the scale variation.

In [10]:
miner.run(
    sample_benchmark="sm",
    mg_directory=mg_dir,
    mg_process_directory="./mg_processes/signal_systematics",
    proc_card_file="cards/proc_card_signal.dat",
    param_card_template_file="cards/param_card_template.dat",
    run_card_file="cards/run_card_signal_small.dat",
    log_directory="logs/signal",
    # systematics=["signal_norm", "scales"],
    systematics=["scales"],
)

15:14 madminer.utils.inter INFO    Generating MadGraph process folder from cards/proc_card_signal.dat at ./mg_processes/signal_systematics
15:14 madminer.utils.inter INFO    Calling MadGraph: /madminer/software/MG5_aMC_v2_9_16/bin/mg5_aMC /tmp/generate.mg5
15:14 madminer.core.madmin INFO    Run 0
15:14 madminer.core.madmin INFO      Sampling from benchmark: sm
15:14 madminer.core.madmin INFO      Original run card:       cards/run_card_signal_small.dat
15:14 madminer.core.madmin INFO      Original Pythia8 card:   None
15:14 madminer.core.madmin INFO      Original config card:    None
15:14 madminer.core.madmin INFO      Copied run card:         madminer/cards/run_card_0.dat
15:14 madminer.core.madmin INFO      Copied Pythia8 card:     None
15:14 madminer.core.madmin INFO      Copied config card:      None
15:14 madminer.core.madmin INFO      Param card:              madminer/cards/param_card_0.dat
15:14 madminer.core.madmin INFO      Reweight card:           madminer/cards/reweight_car

In [11]:
miner.run(
    sample_benchmark="sm",
    mg_directory=mg_dir,
    mg_process_directory="./mg_processes/bkg_systematics",
    proc_card_file="cards/proc_card_background.dat",
    param_card_template_file="cards/param_card_template.dat",
    run_card_file="cards/run_card_background_small.dat",
    log_directory="logs/background",
    # systematics=["bkg_norm", "scales"],
    systematics=["scales"],
)

15:16 madminer.utils.inter INFO    Generating MadGraph process folder from cards/proc_card_background.dat at ./mg_processes/bkg_systematics
15:16 madminer.utils.inter INFO    Calling MadGraph: /madminer/software/MG5_aMC_v2_9_16/bin/mg5_aMC /tmp/generate.mg5
15:16 madminer.core.madmin INFO    Run 0
15:16 madminer.core.madmin INFO      Sampling from benchmark: sm
15:16 madminer.core.madmin INFO      Original run card:       cards/run_card_background_small.dat
15:16 madminer.core.madmin INFO      Original Pythia8 card:   None
15:16 madminer.core.madmin INFO      Original config card:    None
15:16 madminer.core.madmin INFO      Copied run card:         madminer/cards/run_card_0.dat
15:16 madminer.core.madmin INFO      Copied Pythia8 card:     None
15:16 madminer.core.madmin INFO      Copied config card:      None
15:16 madminer.core.madmin INFO      Param card:              madminer/cards/param_card_0.dat
15:16 madminer.core.madmin INFO      Reweight card:           madminer/cards/reweigh

## 4. Load events from LHE file

When adding LHE or Delphes files, use the `systematics` keyword to list which systematic uncertainties apply to which sample:

In [12]:
lhe = LHEReader("data/setup_systematics.h5")

lhe.add_sample(
    lhe_filename="mg_processes/signal_systematics/Events/run_01/unweighted_events.lhe.gz",
    sampled_from_benchmark="sm",
    is_background=False,
    k_factor=1.1,
    # systematics=["signal_norm", "scales"],
    systematics=["scales"],
)

lhe.add_sample(
    lhe_filename="mg_processes/bkg_systematics/Events/run_01/unweighted_events.lhe.gz",
    sampled_from_benchmark="sm",
    is_background=True,
    k_factor=1.1,
    # systematics=["bkg_norm", "scales"],
    systematics=["scales"],
)

15:25 madminer.utils.inter INFO    HDF5 file does not contain nuisance parameters information
15:25 madminer.utils.inter INFO    HDF5 file does not contain finite difference information
15:25 madminer.utils.inter INFO    HDF5 file does not contain observables information
15:25 madminer.utils.inter INFO    HDF5 file does not contain sample summary information
15:25 madminer.utils.inter INFO    HDF5 file does not contain sample information
15:25 madminer.lhe.lhe_rea DEBUG   Adding event sample mg_processes/signal_systematics/Events/run_01/unweighted_events.lhe.gz
15:25 madminer.lhe.lhe_rea DEBUG   Adding event sample mg_processes/bkg_systematics/Events/run_01/unweighted_events.lhe.gz


The next steps are unaffected by systematics.

In [13]:
# Partons giving rise to jets
particles = [
    *Particle.findall(lambda p: p.pdgid.is_quark),
    *Particle.findall(pdg_name="g"),
]

lhe.set_smearing(
    pdgids=[int(p.pdgid) for p in particles],
    energy_resolution_abs=0.0,
    energy_resolution_rel=0.1,
    pt_resolution_abs=None,
    pt_resolution_rel=None,
    eta_resolution_abs=0.1,
    eta_resolution_rel=0.0,
    phi_resolution_abs=0.1,
    phi_resolution_rel=0.0,
)

lhe.add_observable(
    "pt_j1",
    "j[0].pt",
    required=False,
    default=0.0,
)
lhe.add_observable(
    "delta_phi_jj",
    "j[0].deltaphi(j[1]) * (-1.0 + 2.0 * float(j[0].eta > j[1].eta))",
    required=True,
)
lhe.add_observable(
    "met",
    "met.pt",
    required=True,
)

lhe.add_cut("(a[0] + a[1]).m > 122.0")
lhe.add_cut("(a[0] + a[1]).m < 128.0")
lhe.add_cut("pt_j1 > 30.0")

15:25 madminer.lhe.lhe_rea DEBUG   Adding optional observable pt_j1 = j[0].pt with default 0.0
15:25 madminer.lhe.lhe_rea DEBUG   Adding required observable delta_phi_jj = j[0].deltaphi(j[1]) * (-1.0 + 2.0 * float(j[0].eta > j[1].eta))
15:25 madminer.lhe.lhe_rea DEBUG   Adding required observable met = met.pt
15:25 madminer.lhe.lhe_rea DEBUG   Adding cut (a[0] + a[1]).m > 122.0
15:25 madminer.lhe.lhe_rea DEBUG   Adding cut (a[0] + a[1]).m < 128.0
15:25 madminer.lhe.lhe_rea DEBUG   Adding cut pt_j1 > 30.0


In [14]:
lhe.analyse_samples()
lhe.save("data/lhe_data_systematics.h5")

15:25 madminer.lhe.lhe_rea INFO    Analysing LHE sample mg_processes/signal_systematics/Events/run_01/unweighted_events.lhe.gz: Calculating 3 observables, requiring 3 selection cuts, using 0 efficiency factors, associated with systematicsscales
15:25 madminer.lhe.lhe_rea DEBUG   Extracting nuisance parameter definitions from LHE file
15:25 madminer.utils.inter DEBUG   Parsing nuisance parameter setup from LHE file at mg_processes/signal_systematics/Events/run_01/unweighted_events.lhe.gz
15:25 madminer.utils.inter DEBUG   Systematics setup: OrderedDict([('scales', Systematic(name='scales', type=<SystematicType.SCALE: 'scale'>, value='0.5,1.0,2.0', scale=<SystematicScale.MU: 'mu'>))])
15:25 madminer.utils.inter DEBUG   3 weight groups
15:25 madminer.utils.inter DEBUG   Extracting nuisance parameter information for systematic scales
15:25 madminer.utils.inter DEBUG   New weight group: Central scale variation
15:25 madminer.utils.inter DEBUG   Weight group identified as scale variation
15:

RuntimeError: Nuisance parameter scales_nuisance_param_0 does not exist

### A look at distributions

The function `plot_systematics()` makes it easy to check the effect of the various nuisance parameters on a distribution:

In [ ]:
_ = plot_systematics(
    filename="data/lhe_data_systematics.h5",
    theta=np.array([0.0, 0.0]),
    observable="pt_j1",
    obs_label="$p_{T,j}$",
    obs_range=(30.0, 400.0),
    ratio_range=(0.7, 1.4),
)

14:52 madminer.analysis.da INFO    Loading data from data/lhe_data_systematics.h5
14:52 madminer.utils.inter INFO    HDF5 file does not contain nuisance parameters information
14:52 madminer.utils.inter INFO    HDF5 file does not contain finite difference information
14:52 madminer.analysis.da INFO    Found 2 parameters
14:52 madminer.analysis.da INFO      0: CWL2 (LHA: dim6 2, Power: 2, Range: (-20.0, 20.0))
14:52 madminer.analysis.da INFO      1: CPWL2 (LHA: dim6 5, Power: 2, Range: (-20.0, 20.0))
14:52 madminer.analysis.da INFO    Did not find nuisance parameters
14:52 madminer.analysis.da INFO    Found 6 benchmarks
14:52 madminer.analysis.da DEBUG      sm: CWL2 = 0.00e+00, CPWL2 = 0.00e+00
14:52 madminer.analysis.da DEBUG      w: CWL2 = 15.20, CPWL2 = 0.10
14:52 madminer.analysis.da DEBUG      neg_w: CWL2 = -1.54e+01, CPWL2 = 0.20
14:52 madminer.analysis.da DEBUG      ww: CWL2 = 0.30, CPWL2 = 15.10
14:52 madminer.analysis.da DEBUG      neg_ww: CWL2 = 0.40, CPWL2 = -1.53e+01
14:52 m

ValueError: operands could not be broadcast together with shapes (15,) (0,) 

In [ ]:
_ = plot_systematics(
    filename="data/lhe_data_systematics.h5",
    theta=np.array([0.0, 0.0]),
    observable="delta_phi_jj",
    obs_label=r"$\Delta\phi$",
    obs_range=(-3.1, 3.1),
    ratio_range=(0.7, 1.4),
)

## 5. Sampling

Let's generate training data for the SALLY method.

In [ ]:
sampler = SampleAugmenter("data/lhe_data_systematics.h5", include_nuisance_parameters=True)

x, theta, t_xz, _ = sampler.sample_train_local(
    theta=sampling.benchmark("sm"),
    n_samples=100000,
    folder="./data/samples",
    filename="train_score_systematics",
)

## 6. Training

The SALLY estimator will learn the score in terms of both the physics parameters and the nuisance parameters. In our case, its output will therefore be a vector with five components.

In [ ]:
estimator = ScoreEstimator(n_hidden=(50,))

estimator.train(
    method="sally",
    x="data/samples/x_train_score_systematics.npy",
    t_xz="data/samples/t_xz_train_score_systematics.npy",
)

estimator.save("models/sally_systematics")

## 7. Fisher information

The Fisher information is now also a 5x5 matrix. Constraint terms on the nuisance parameters, representing our prior knowledge on their values, can be calculated with `FisherInformation.nuisance_constraint_information()`.

In [ ]:
fisher = FisherInformation("data/lhe_data_systematics.h5")

In [ ]:
?fisher.full_information

In [ ]:
info_sally, _ = fisher.full_information(
    theta=np.zeros(5),
    model_file="models/sally_systematics",
    luminosity=1000000.0,
    include_xsec_info=False,
)

In [ ]:
info_nuisance = fisher.nuisance_constraint_information()
info = info_sally + info_nuisance

In [ ]:
for row in info:
    print(" ".join(["{:6.2f}".format(entry) for entry in row]))

### Fisher contours and profiled information

When looking at a subset of these five parameter, we can either ignore the other parameters ("project"), or conservatively pick the most pessimistic value of them ("profile" them). MadMiner provides the functions `profile_information()` and `project_information()` for this purpose. Let's do that for the parameter space of the two physics parameters:

In [ ]:
info_proj = project_information(info, [0, 1])
info_prof = profile_information(info, [0, 1])

In [ ]:
_ = plot_fisher_information_contours_2d(
    [info_proj, info_prof],
    inline_labels=["Projected", "Profiled"],
    xrange=(-0.1, 0.1),
    yrange=(-0.1, 0.1),
)

For illustration, we can look at just one physics parameter and one nuisance parameter:

In [ ]:
info_demo = project_information(info, [0, 4])

info_demo_profiled = np.zeros((2, 2))
info_demo_profiled[0, 0] = profile_information(info_demo, [0])

In [ ]:
_ = plot_fisher_information_contours_2d(
    [info_demo, info_demo_profiled],
    inline_labels=[None, "Profiled"],
    xrange=(-0.2, 0.2),
    yrange=(-0.6, 0.6),
    xlabel="Parameter of interest",
    ylabel="Nuisance parameter",
)